<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

Votre tâche pour ce laboratoire sera d'écrire un filtre de Kalman étendu (EKF) pour la localisation basée sur une carte.

# Définition de la carte

Nous devons saisir les coordonnées exactes des points de repère utilisés pour la localisation. Ces coordonnées sont lues depuis [le fichier de carte](../../packages/ekf_localization/map.yaml). Actuellement, il s'agit du fichier de carte utilisé dans la Duckiematrix. Vous trouverez également, dans le même répertoire, un fichier de carte pour [la carte utilisée dans AA341](../../packages/ekf_localization/aa3341.yaml). Pour tester la carte utilisée au AA3341, vous devrez modifier le [fichier launch](../../packages/ekf_localization/launch/ekf_localization_node.launch)  pour prendre la nouvelle spécifications cartographique (c'est-à-dire, changer l'argument `map_name` de `map` à `aa3341`).  La spécification de la carte a le format suivant :

```yaml
map:
  "1":
    position: [0.6786, 1.76085]
  "20":
    position: [-0.02925, 1.8252]
  ...
```

Les nombres, par exemple "1", correspondent aux identifiants des [AprilTags](https://docs.wpilib.org/en/stable/docs/software/vision-processing/apriltag/apriltag-intro.html). Il est également important de bien définir l'origine du repère du monde. Dans la Duckiematrix, le canard jaune se trouve initialement à l'origine du repère du monde, pointant dans la direction de l'axe y positif et de l'axe x positif vers la droite.

# Définition des paramètres du filtre

Votre EKF nécessite la définition de [certains paramètres dans le fichier](../../packages/ekf_localization/config/ekf_localization_node/default.yaml). Il s'agit notamment de la pose initiale et de la covariance du Duckiebot, ainsi que des covariances de processus et de mesure. Si vous utilisez la Duckiematrix, vous pouvez probablement ignorer ces paramètres, car la pose initiale du robot correspond à l'emplacement du Duckiebot dans la Duckiematrix. Dans Duckietown, vous devrez veiller à placer initialement le Duckiebot approximativement à la position indiquée dans votre configuration. 

**Remarque** : Vous devez remettre le Duckiebot dans sa position initiale à chaque fois que vous testez votre code. Dans la Duckiematrix, vous pouvez le faire en appuyant sur la touche `R`.


# Mise en œuvre de votre EKF

Tout le code que vous devrez écrire ira dans le fichier [ekf.py](../../packages/solution/ekf.py) (bien que, si vous êtes intéressés, les détails sur la façon dont le [nœud ROS](../../packages/ekf_localization/src/ekf_localization_node.py) est implémenté se trouvent dans le lien ci-présent).

La pose du Duckiebot est stockée dans `self.q` et est un tableau numpy de la forme suivante :
$$
[x, y, \theta].
$$

La covariance d'état du Duckiebot est stockée dans `self.P` et possède la structure suivante :

$$
P = 
\begin{bmatrix}
P_{xx} &  P_{xy} & P_{x\theta} \\
P_{xy} & P_{yy} & P_{y\theta} \\
p_{x\theta} & P_{y\theta} & P_{\theta\theta}
\end{bmatrix}.
$$

Il y a deux fonctions à écrire : `predict` et `update`. 


## L'étape de prédiction

Le fonction `predict`:

```python
def predict(self, dX, dT):
```

Cette fonction prend en entrée `dX`, qui représente le déplacement linéaire du Duckiebot vers l'avant depuis le dernier appel (en mètres), et `dT`, qui représente l'angle de rotation du Duckiebot depuis le dernier appel (en radians). Ces valeurs sont calculées à partir des données de l'encodeur et servent d'approximation à notre commande.

La covariance du modèle de processus est stockée dans `self.Q` et possède la structure suivante :

$$
Q = 
\begin{bmatrix}
Q_{xx} & 0 \\
0 & Q_{\theta\theta}
\end{bmatrix},
$$

où $Q_{xx}$ et $Q_{\theta\theta}$ sont des covariances spécifiées [dans le fichier de configuration](../../packages/ekf_localization/config/ekf_localization_node/default.yaml).

La mise en œuvre de la fonction predict comporte trois étapes :

### Étape 1 : Propager l'état

```python
# Étape 1 : mettre à jour l'estimation de la pose à l'aide du modèle cinématique
# TODO: à faire
self.q[0] = self.q[0]
self.q[1] = self.q[1]
self.q[2] = self.q[2]
```
La première étape consiste à propager l'estimation d'état à l'aide du modèle de mouvement sans bruit $f$. Nous pouvons utiliser notre modèle cinématique pour cela (vous vous en souvenez ?).

**Remarque** : Veillez à appeler la fonction `angle_wrap` à chaque modification de l’orientation. L’orientation doit toujours se situer dans l’intervalle $[-\pi,\pi]$.

### Étape 2 : Calcul des jacobiennes du modèle de processus

```python
# Étape 2 : Calcul des jacobiennes du modèle de processus
# TODO: Définition de F et W
F = np.array([])
W = np.array([])
```

Il nous faut ensuite linéariser le modèle de processus. Pour ce faire, nous devons calculer : $F = \frac{\partial f}{\partial x}$ et $W = \frac{\partial f}{\partial w}$. Comme $f$ comporte trois équations et que `self.q` possède trois états, la matrice $F$ sera de dimension $3\times 3$. Comme il n'y a que deux entrées de "contrôle" ($dX$ et $dT$) et que nous supposons que le bruit est additif sur ces entrées, la matrice $W$ sera de dimension $3\times 2$.


### Étape 3 : Mise à jour de la covariance d'état

```python
# Étape 3: Faire l'estimation de covariance de l'état
# TODO: à faire
self.P = self.P
```

Enfin, nous devons mettre à jour notre prédiction de la covariance d'état en utilisant les jacobiennes que nous venons de calculer ainsi que la covariance du bruit de processus.


## L'étape de correction (`update`)

Les mesures des points de repère (AprilTags) comportent deux composantes : la distance et l’orientation. Il est important de noter que ces mesures sont relatives à la position actuelle du robot, puisque la caméra est fixée à celui-ci.


Le fonction `update`:

```python
    def update(self, z: np.ndarray, tag_xy: np.ndarray):
```

prend en entrée `z` qui est un array 2D contenant $[r, \phi]$ où $r$ est la portée et $\phi$ est le relèvement, et `tag_xy` qui contient une position 2D $[l_x, l_y]$ du point de repère détecté (que nous lisons à partir du [fichier de carte](../../packages/ekf_localization/map.yaml)).


### Étape 1 : Calculer les mesures prévues

```python
# Étape 1: calculer les mesures de portée et de relèvement prévues
# TODO: mettre à jours les équations suivants
rng_pred = 1.0
bearing_pred = 0.0
z_pred = np.array([rng_pred, bearing_pred])
```

La première étape consiste à utiliser notre modèle de mesure, `h`, pour calculer les prédictions (sans bruit) des mesures. Pour ce faire, nous pouvons utiliser notre estimation de l'état actuel et la position connue de l'étiquette. Il vous faudra donc dériver les modèles de mesure. Autrement dit, connaissant l'état actuel du robot et la position du point de repère, quelles devraient être la portée ? Quel devrait être le relèvement ? Pour le calcul du relèvement, veillez tout particulièrement à le calculer **dans le repère actuel du robot**.


### Étape 2 : Calculer l'innovation

```python
# Étape 2 : Calculer l'innovation
# TODO: Trouver l'innovation y
y = np.array([0.0, 0.0])
```

The innovation is the difference between the measurements we actually received (`z`) and the ones that we predicted using our measurement model (`z_pred`).

### Étape 3 : Calculer le jacobien du modèle de mesure

```python
# Étape 3 : Calculer le jacobien du modèle de mesure
# TODO: Trouver H
H = np.array([])
```

Il nous faut ensuite linéariser le modèle de mesure. Pour ce faire, nous devons calculer : $H = \frac{\partial h}{\partial x}$. Comme nous avons deux mesures et trois états, cette matrice sera de dimension $2\times 3$.


### Étape 4 : Calculer le gain de Kalman

```python
# Étape 4 : Calculer le gain de Kalman
# TODO: Trouver K
K = np.array([])
```

Ensuite, nous calculons le gain de Kalman, `K`, qui est utilisé pour pondérer nos estimations précédentes par rapport aux informations fournies par ces nouvelles mesures.


### Étape 5 : Mise à jour des estimations de pose et de covariance de pose

```python
# Étape 5 : Mise à jour des estimations de pose et de covariance de pose
# TODO: modifier les équations
self.q = self.q
self.q[2] = wrap_angle(self.q[2])
self.P = self.P
```

Enfin, utilisez l'innovation, le gain de Kalman et le jacobien de mesure pour mettre à jour l'estimation de la pose et de la covariance associée à l'aide des équations EKF.


# Tests et débogage

Une fois l'implémentation terminée, vous pouvez tester votre code en suivant la méthode décrite dans le fichier [README](../../README.md). Si votre estimateur fonctionne correctement, l'état réel devrait rester à l'intérieur de l'ellipse de covariance de l'estimation produite lorsque vous pilotez votre véhicule (vous pouvez utiliser le joystick dans noVNC). Si vous utilisez Duckiematrix, l'état réel peut être visualisé. Si vous utilisez un véritable Duckiebot, vous devrez effectuer l'estimation vous-même.

Si votre implémentation contient une erreur, votre estimation risque de diverger très rapidement. Le débogage peut s'avérer complexe, car la divergence est rapide et il est difficile d'en déterminer la cause. Isoler les étapes de prédiction et de mise à jour peut être utile. Plus précisément, vous pouvez tester chaque étape indépendamment en définissant `no_update: True` ou `no_predict: True` dans le [fichier de configuration](../../packages/ekf_localization/config/ekf_localization_node/default.yaml). Soyez toutefois prudent avec `no_predict: True`, car vous ne recevez aucune information sur les mouvements du robot et vous vous basez entièrement sur les mesures des points de repère. Si vous observez une interruption des mesures des points de repère, votre estimation ne sera pas mise à jour. Par conséquent, si vous constatez que votre estimateur ne fonctionne pas correctement, il est recommandé de commencer avec `no_update: True` jusqu'à ce que vous soyez certain de la justesse de votre prédiction, puis de définir `no_update: False` (en laissant `no_predict` à `False`).

Si vous avez toujours des difficultés à comprendre l'origine du problème, il peut être utile d'imprimer les résultats intermédiaires des étapes ci-dessus et de vérifier leur exactitude un par un.

Bonne chance !